# Get text embeddings

Computes CLIP text embeddings for all 40 CelebA attributes using prompt ensembling:
- For each attribute × {pos, neg}: embed all prompt variants separately, average, L2-normalize.
- Result is the canonical text representation used for zero-shot classification.

**Run once** — output is shared across all zero-shot evaluation notebooks.

| output | schema |
|---|---|
| `data/text_embeddings.parquet` | `attribute (str), label (str), 0..767 (float16)` |

`label` ∈ {`pos`, `neg`}. Embeddings are L2-normalized (unit vectors).

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from transformers import AutoModel, AutoTokenizer

load_dotenv()

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'pyproject.toml').exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from software.prompts import PROMPTS

In [ ]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
MODEL_ID            = 'openai/clip-vit-large-patch14'
PARQUET_COMPRESSION = 'snappy'

OUT_PATH = ROOT / 'data' / 'text_embeddings.parquet'

HF_TOKEN = os.getenv('HF_TOKEN')
device   = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype    = torch.bfloat16 if device == 'cuda' else torch.float32

print(f'Attributes : {len(PROMPTS)}')
print(f'Device     : {device} | dtype: {dtype}')
print(f'Output     : {OUT_PATH}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model     = AutoModel.from_pretrained(
    MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True, token=HF_TOKEN,
).to(device).eval()
print(f'Model loaded: {MODEL_ID}')

In [ ]:
records = []

with torch.no_grad():
    for attribute, variants_dict in PROMPTS.items():
        for label, texts in variants_dict.items():  # label = 'pos' or 'neg'
            inputs = tokenizer(
                texts, padding=True, truncation=True, return_tensors='pt'
            ).to(device)
            embs = model.get_text_features(**inputs).float().cpu().numpy()  # (n_variants, 768)

            # #10 — prompt ensembling on the unit hypersphere:
            #   1) L2-normalize each prompt-variant embedding,
            #   2) average,
            #   3) L2-normalize the mean.
            # This avoids variants with larger raw norms dominating the average
            # and keeps the result on the unit sphere where cosine similarity is defined.
            norms = np.linalg.norm(embs, axis=1, keepdims=True) + 1e-12
            embs_normalized = embs / norms
            mean = embs_normalized.mean(axis=0)
            avg  = mean / (np.linalg.norm(mean) + 1e-12)

            row = {'attribute': attribute, 'label': label}
            row.update({str(i): float(avg[i]) for i in range(len(avg))})
            records.append(row)

print(f'Computed embeddings for {len(records)} (attribute, label) pairs')

In [ ]:
df = pd.DataFrame(records)
feat_cols = [str(i) for i in range(768)]
df[feat_cols] = df[feat_cols].astype('float16')

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT_PATH, compression=PARQUET_COMPRESSION, index=False)

print(f'Saved {len(df)} rows → {OUT_PATH}')
print(f'Attributes: {df["attribute"].nunique()} | Labels: {df["label"].unique()}')
df[['attribute', 'label']].head(8)